### Confidence intervals based on EM and Variational Bayes algorithms.
### Simulation exercises for a Mixture-of-Gaussians model. 

#### Introduction
This Notebook performs simulations where the classical Expectation-Maximization (EM) algorithm and the Variational Bayes (VB) algorithm are used to fit a Mixture-of-Gaussians model. EM is a maximum likelihood algorithm and regards **parameters as scalars**, whereas in the Bayesian approach a **whole distribution for each parameter** of the model is fitted. 

The main concern of the simulation is to assess uncertainty in parameter estimates in each approache. Both methods can quantify uncertainty, but the resulting intervals answer different questions:

- An **EM confidence interval** describes the repeated-sampling variability of the maximum-likelihood estimator.
- A **VB credible interval** contains a specified amount of probability under the approximate posterior distribution, conditional on the observed data and the prior.

**Goals**
- Assess the correctness of the implementation in the package AAIMF created for this project.
- Compare how the classical and Bayesian approaches quantify uncertainty in parameter estimates and their interpretation.

#### Classical and Bayesian measures of uncertainty

**Classical confidence intervals** based on EM are obtained upon the observed information using an unconstrained parameterization:

\\[
\boldsymbol{\theta} = \left( 
\eta,\, \boldsymbol{\mu}_1,\, \boldsymbol{\mu}_2,\, 
\log\boldsymbol{\lambda}_1,\, \log\boldsymbol{\lambda}_2 
\right),\quad \eta=\log\frac{\pi_2}{\pi_1}
\\]

\\[
\widehat{\operatorname{Var}}(\widehat{\boldsymbol{\theta}}) =
\left[ -\nabla^2\ell(\widehat{\boldsymbol{\theta}}) \right]^{-1}.
\\]

The covariance matrix \\( \widehat{\operatorname{Var}}(\widehat{\boldsymbol{\theta}}) \\) is approximated as the inverse of the numerical Hessian of the negative log-likehood with unconstrained probability and variance parameters; i.e. the Hessian of 
\\(
\operatorname{NLL}(\boldsymbol\theta) = -\ell 
\left( \operatorname{logit}^{-1}(\eta), \boldsymbol\mu, \exp(\boldsymbol\xi) \right)
\\).

Wald intervals for the \\(1-\alpha\\) level of confidence are constructed on the unconstrained scale and then are transformed back to probabilities and variances:

\\[
\widehat{\theta} \pm z_{1-\alpha/2} \sqrt{ \widehat{\operatorname{Var}}(\widehat{\boldsymbol{\theta}}) }.
\\]

An alternative approach could be to work directly on the constrained parameterization and take into account the Jacobian of the parameter transformations when computing the covariance matrix. However, the approach described above is considered appropriate for the following reasons: 
- Since the Hessian is evaluated locally in the neighborhood of the fitted parameters (which satisfy the contraints), the small parameter perturbations used to compute the numerical Hessian are expected to remain under the admissible parameter region.
- If any of the fitted parameters lie close to a constraint boundary, then the Hessian may not be stable around the fitted value. However, in simulations run in another context (maximum likelihood of structural time series models, not reported here), restricting the perturbations to the feasible region, did not resolve the stability issue.

**VB credible intervals** are obtained upon the variational factors:

\\[
q(\pi_1) =
\operatorname{Beta} \left( \alpha_1,\sum_{j\ne1}\alpha_j \right),
\quad
q(\mu_{kd}) = t_{2a_{kd}}
\left( m_{kd}, \sqrt{\frac{b_{kd}}{a_{kd}\beta_k}} \right),
\quad
q(\lambda_{kd}) = F_\Gamma^{-1}(p).
\\]

Thus, credible intervals for the means use Student-\\(t\\) quantiles, while precision intervals are obtained using Gamma quantiles directly.

Further details on the computation of confidence intervals are given in the comments of the source code of the AAIMF R-package: `AAIMF/R/confint-avem.R` and `AAIMF/R/mog-normal-gamma.R`.

### Load AAIMF R package, auxiliary libraries and functions

In [1]:
pkg_lib <- "/home/jupyter-aai26_javier.lopez-7483b/.conda/envs/r-env/lib/R/library"
install.packages("../../AAIMF_0.1.8.tar.gz", lib = pkg_lib, repos = NULL, type = "source", verbose=FALSE)
library(AAIMF)
packageVersion("AAIMF")

[1] ‘0.1.8’

In [2]:
library(AAIMF)
library(future)
library(future.apply)
library(progressr)

source("../fn_auxiliary.R")
source("./fn-sim-mog.R")

### Load workspace saved in a previous session
At the end of the Notebook, the workspace is saved to an .RData file. See comments in Section "Save current workspace/Load workspace".

- **sim-ci-prior-1.RData**: Loads the workspace saved after running the simulation with the default priors (see discussion below).
- **sim-ci-prior-2.RData**: Loads the workspace saved after running the simulation with the priors `vb_priors <- list(alpha0 = 1, beta0 = 0.01, m0 = c(0, 0), a0 = 1, b0 = 1)`.

In [3]:
env1 <- new.env(parent = globalenv())
load(file = "./sim-ci-prior-1.RData", envir = env1)
length(env1$res_list)

env2 <- new.env(parent = globalenv())
load(file = "./sim-ci-prior-2.RData", envir = env2)
length(env2$res_list)

[1] 5000

[1] 5000

### Choose simulation parameters

In [4]:
# number of simulation iterations and seed

niter <- 5000
seed <- 123

# data generating process (DGP)

dgp <- list(
  N = 75,
  pis = c(0.35, 0.65),
  mus = rbind(c(-2, -1), c(2, 1)),
  #Sigmas = list(diag(c(0.4, 0.8)), matrix(c(0.8, 0.3, 0.3, 0.6), 2, 2)),
  Sigmas = list(diag(c(0.4, 0.8)), diag(c(0.8, 0.6))))

# model
# NOTE predictive densities are implemented only for 'mog_diagonal_cov'

model_family <- 'mog_diagonal_cov' # 'mog_full_cov'

vb_model_family = switch(model_family,
    'mog_diagonal_cov' = 'mog_normal_gamma',
    'mog_full_cov' = 'mog_normal_wishart')

# priors for variational Bayes

vb_priors <- list() # default values
#vb_priors <- list(alpha0 = 1, beta0 = 0.01, m0 = c(0, 0), a0 = 1, b0 = 1)

# number of cores used in the parallel execution
                      
available_workers <- parallel::detectCores()
nworkers <- 50 #parallel::detectCores() - 1
cat(paste(sprintf("Using %d cores out of %d available cores.", nworkers, available_workers)))

Using 50 cores out of 256 available cores.

#### Further auxiliary elements and sanity checks

In [4]:
tmp <- dgp_sanity_checks(dgp, model_family)

dgp$Lambdas <- tmp$Lambdas
dgp_pars_nms <- tmp$dgp_pars_nms
dgp_vec_pars <- tmp$dgp_vec_pars
dgp_struct <- tmp$dgp_struct

rm(tmp)

OK DGP definition.
OK The order of the parameters in the DGP specification and in the model match each other.


In [5]:
dgp$Lambdas

[[1]]
[1] 2.50 1.25

[[2]]
[1] 1.250000 1.666667

In [6]:
# fit one model just to generate 'struct_mod'
dgp_data <- mog_sim(N = 50, pis = dgp$pis, mus = dgp$mus, Sigmas = dgp$Sigmas)
model0 <- em_init(make_model(model_family, dgp_data$X, dgp_data$K))
fit0 <- avem_run(model0, target = "loglik", maxiter = 500, tol = 1e-8, verbose = FALSE, debug = FALSE)
struct_mod <- get_pars_structure_v2(fit0$model$pars)
rm(dgp_data, model0, fit0)
print(names(struct_mod))

[1] "pis"     "mus"     "Lambdas"


### Computations for one iteration of the simulation

In [7]:
one_iteration <- function(i)
{
  # Generate data

  dgp_data <- mog_sim(N = dgp$N, pis = dgp$pis,
    mus = dgp$mus, Sigmas = dgp$Sigmas)
    #seed = 123 # handled by clusterSetRNGStream()

  # ------------ ------------ ------------ ------------ ------------ ------------
  # EM (loglik)
  # avem_run() returns ordered parameters, no need to handle it here.
  # ------------ ------------ ------------ ------------ ------------ ------------

  model1 <- make_model(model_family, dgp_data$X, dgp_data$K)

  model1 <- em_init(model1, method = "kmeans")

  fit1 <- avem_run(model1, target = "loglik",
    maxiter = 500, tol = 1e-8, verbose = FALSE, debug = FALSE)

  point_bias1 <- unlist(fit1$model$pars) - dgp_vec_pars
  
  #suppressWarnings({ ci1 <- confint(fit1, level = 0.95) })
  ci1 <- confint(fit1, level = 0.95)

  ci1_info <- ci_get_info(ci1, struct_mod, dgp_vec_pars, dgp_struct) #debug = TRUE)
  ci1_coverage <- ci1_info$coverage
  ci1_widths <- ci1_info$width

  ci1_warnings <- ci1$warnings
  if (ci1_warnings > 0)
  {
    ci1_coverage[] <- NA
    ci1_widths[] <- NA
  }

  z1 <- max.col(fit1$model$rprobs)
  accuracy1 <- mean(z1 == dgp_data$states)
  
  # ------------ ------------ ------------ ------------ ------------ ------------
  # Variational Bayes.
  # ------------ ------------ ------------ ------------ ------------ ------------

  model2 <- make_model(family = vb_model_family, dgp_data$X, dgp_data$K,
    priors = vb_priors)

  model2 <- mog_init(model2, #seed = 123321,
    kmeans_nstart = 10, kmeans_prob = 0.95, #args = list(),
    debug = FALSE)

  fit2 <- mog_cavi(model2, maxiter = 500, tol = 1e-8,
    verbose = FALSE, debug = FALSE)

  tmp <- order_mog_ng_pars(fit2$model$pars, fit2$model$rprobs)
  fit2$model$pars <- tmp$opars
  fit2$model$rprobs <- tmp$orprobs

  # remember updating posteriors!! 
  # compute them anew is simpler, alternatively use tmp$opars
  fit2$posteriors <- mog_ng_posteriors(fit2$model)

  #point_bias2 <- unlist(diff_lists(fit2$posteriors, dgp))
  point_bias2 <- unlist(fit2$posteriors[dgp_pars_nms]) - dgp_vec_pars

  ci2 <- confint(fit2, level = 0.95, struct = struct_mod)

  ci2_info <- ci_get_info(ci2, struct_mod, dgp_vec_pars, dgp_struct) #debug = FALSE)
  ci2_coverage <- ci2_info$coverage
  ci2_widths <- ci2_info$width

  z2 <- max.col(fit2$model$rprobs)
  accuracy2 <- mean(z2 == dgp_data$states)

  vb_pars_probs <- mog_ng_posterior_probs(model = fit2$model, B = 5000)

  # ------------ ------------ ------------ ------------ ------------ ------------
  # Predictive log-density comparison on an independent test set.
  # ------------ ------------ ------------ ------------ ------------ ------------

  dgp_test <- mog_sim(N = dgp$N, pis = dgp$pis, mus = dgp$mus, Sigmas = dgp$Sigmas)

  # EM plug-in predictive density:
  # p_EM(x_new) = p(x_new | theta_hat_EM)

  # old version
  #em_lpd_i <- dmog_diag_logdens(X = dgp_test$X,
  #  pis = fit1$model$pars$pis, mus = fit1$model$pars$mus,
  #  Lambdas = fit1$model$pars$Lambdas)
  em_lpd_i <- log_predictive_density(fit1$model, dgp_test$X)

  EM_test_lpd_mean <- mean(em_lpd_i)

  # VB posterior predictive density:
  # p_VB(x_new) = E_q[p(x_new | theta)]
  #
  # The expectation is approximated drawing from the posterior q.

  #vb_lpd_i <- mog_ng_posterior_predictive_logdens(
  #  Xtest = dgp_test$X, model = fit2$model, B = 1000)
  vb_lpd_i <- log_predictive_density(fit2$model, dgp_test$X, 1000)
  
  VB_test_lpd_mean <- mean(vb_lpd_i)

  predictive_measures <- c(
    EM_test_lpd_mean = EM_test_lpd_mean,
    VB_test_lpd_mean = VB_test_lpd_mean,
    VB_minus_EM_test_lpd_mean = VB_test_lpd_mean - EM_test_lpd_mean)

  list(point_bias1 = point_bias1, point_bias2 = point_bias2,
    coverage = c(EM = ci1_coverage, VB = ci2_coverage),
    widths = c(EM = ci1_widths, VB = ci2_widths),
    warnings = ci1_warnings,
    accuracy = c(EM = accuracy1, VB = accuracy2),
    vb_pars_probs = vb_pars_probs,
    predictive = predictive_measures)
}

#### Run test example

In [8]:
test_run <- one_iteration(1)

In [10]:
#print(test_run)

### Parallel execution of the simulation

In [9]:
future::plan(multicore, workers = nworkers)

#handlers(global = TRUE)
handlers("txtprogressbar")

res_list <- with_progress({
  p <- progressor(along = seq_len(niter))

  future_lapply(
    X = seq_len(niter),
    FUN = function(i) {
      res = one_iteration(i)
      p() # update progress
      res
    },
    future.seed = seed
  )
})

future::plan(future::sequential) # back to sequential processing

In [11]:
#print(res_list[[1]])

In [6]:
cat(paste("Number of warnings or issues in all iterations (0 warnings)."))
#table(do.call("rbind", lapply(res_list, function(x) x$warnings)))
tmp <- evalq({
  table(do.call("rbind", lapply(res_list, function(x) x$warnings)))
}, envir = env1)
tmp

tmp <- evalq({
  table(do.call("rbind", lapply(res_list, function(x) x$warnings)))
}, envir = env2)
tmp

Number of warnings or issues in all iterations (0 warnings).


   0 
5000 


   0 
5000 

### Results

In [15]:
#names(res_list[[1]])
evalq({ names(res_list[[1]]) }, envir = env2)

[1] "point_bias1"   "point_bias2"   "coverage"      "widths"       
[5] "warnings"      "accuracy"      "vb_pars_probs" "predictive"

#### Accuracy

Both procedures reach an average classification accuracy of 0.997. The data generating process consists of two mixture components that are relatively well separated. Consequently, the latent allocations can be identified almost perfectly. In this simulation, there is no meaningful difference between EM and VB (or between the choices of priors) in classification performance.

In [6]:
#res_list[[1]]$accuracy
#round(colMeans(do.call("rbind", lapply(res_list, function(x) x$accuracy))), 3)
tab1 <- evalq({
    colMeans(do.call("rbind", lapply(res_list, function(x) x$accuracy)))
}, envir = env1)
round(tab1, 3)

tab2 <- evalq({
    colMeans(do.call("rbind", lapply(res_list, function(x) x$accuracy)))
}, envir = env2)
round(tab2, 3)

EM    VB 
0.997 0.997

EM    VB 
0.997 0.997

#### Parameter estimates. Point bias

The bias for most of the parameter estimates are on average small. EM estimates the mixing probabilities and component means almost without bias, although its precision estimates show systematic positive biases, especially for `Lambdas1` \\(\lambda_{11}\\), whose average bias is 0.349. 

Under the default prior, VB displays a noticeable displacement of some component means toward the prior mean \\(m_0=0\\), especially for \\(\mu_{11}\\), whose average bias is \\(0.086\\), and to a lesser extent for \\(\mu_{21}\\), whose bias is 0.039. 

A large negative bias of -0.716 is obtained for \\(\lambda_{11}\\). With the weaker prior, however, the VB biases become substantially smaller: the bias of \\(\lambda_{11}\\) falls to -0.055, while those of the component means are close to zero. The discussion below would confirm that this bias is an effect induced by the choice of the prior. 

In [19]:
tmp1 <- evalq({
  tab <- rbind(
    colMeans(do.call("rbind", lapply(res_list, function(x) x$point_bias1))),
    colMeans(do.call("rbind", lapply(res_list, function(x) x$point_bias2))))
  colnames(tab) <- names(dgp_vec_pars)
  rownames(tab) <- c("EM", "VB")
  round(tab, 3)
}, envir = env1)
tmp1

tmp2 <- evalq({
  tab <- rbind(
    colMeans(do.call("rbind", lapply(res_list, function(x) x$point_bias1))),
    colMeans(do.call("rbind", lapply(res_list, function(x) x$point_bias2))))
  colnames(tab) <- names(dgp_vec_pars)
  rownames(tab) <- c("EM", "VB")
  round(tab, 3)
}, envir = env2)
tmp2

,pis1,pis2,mus1,mus2,mus3,mus4,Lambdas1,Lambdas2,Lambdas3,Lambdas4
EM,0.000,0.000,0.003,0.003,-0.003,0.000,0.349,0.179,0.088,0.113
VB,0.006,-0.006,0.086,-0.034,0.039,-0.018,-0.716,0.045,-0.054,0.000


,pis1,pis2,mus1,mus2,mus3,mus4,Lambdas1,Lambdas2,Lambdas3,Lambdas4
EM,0.000,0.000,0.003,0.003,-0.003,0,0.349,0.179,0.088,0.113
VB,0.005,-0.005,0.006,0.004,-0.001,0,-0.055,0.116,0.070,0.054


#### Coverage of confidence intervals

The original focus of the design of the simulation is the comparison of confidence intervals. In particular, the table below displays the coverage reached for each parameter, respectively with each method.

In [20]:
tmp <- evalq({
tab <- rbind(colMeans(do.call("rbind", lapply(res_list, function(x) x$coverage)), na.rm=TRUE))
tab <- rbind(tab[,startsWith(colnames(tab), "EM")], tab[,startsWith(colnames(tab), "VB")])
colnames(tab) <- names(dgp_vec_pars)
rownames(tab) <- c("EM", "VB")
round(tab, 3)
}, envir = env1)
tmp

,pis1,pis2,mus1,mus2,mus3,mus4,Lambdas1,Lambdas2,Lambdas3,Lambdas4
EM,0.954,0.954,0.928,0.942,0.930,0.942,0.924,0.929,0.936,0.939
VB,0.953,0.953,0.950,0.955,0.951,0.952,0.721,0.964,0.944,0.955


**Overall assessment of coverage**

With 5000 iterations, the Monte Carlo standard error around 0.95 is approximately
\\( \sqrt{\frac{0.95(0.05)}{5000}}\approx0.003 \\). Thus those deviations from the nominal level 0.95 benyond \\( 0.95\pm 2\times 0.003 = (0.944, 0.956)\\) are statistically meaningful. 

The coverage of EM departs therefore signigicantly more often from the nominal level 0.95. For example it is **0.928 for `mus1`**, **0.930 for `mus3`** and **0.929 for `Lambdas2`**.  No warnings were obtained indicating that the inverse of the numerical Hessian departed from positive-definitiness. Thus, a *good* Hessian alone does not ensure good coverage.

The VB method is closer than EM to the nominal level (except for one case discussed below) and overall performs better in this sense. This is somewhat noteworthy considering that the data generating process consists of relatively clearlly separated components.

For the generating process chosen here, a clear separation of the components does not imply a large effective sample size within every component. The smaller component contains only about \(75(0.35)=26\) observations. 

The performance of EM and VB observed in this particular simulation reflects to some extent the **limitations of Wald intervals in finite (relatively small) samples** and the **ability of the Student-\\(t\\) and Gamma variational marginals to represent broader finite sample distributions** (eg. skewness and heavier tails).

**Low coverage for the precision parameter `Lambdas1` \\(\lambda_{11}\\)**

It is striking the low coverage for the precision parameter `Lambdas1` with VB: 0.721, far from 0.924 by EM. This result was at first understood as a possible error in the implementation. Inspecting the code no issues were found.

A further insight into this apparent anomaly led to the following conclusion. It provides an insight into an issue that was overlooked when designing the simulation.

- The present **simulation is a frequentist coverage experiment**. However, Bayesian intervals need not by construction have a similar coverage behaviour as the the frequentist coverage based on fixed parameters. 

    - The EM interval is designed to attain 95% coverage (at least asymptotically) under repeated sampling.
    - The VB interval contains 95% of an approximate variational posterior marginal under a specified prior. 

- **Default priors may not be suitable.** For the Normal–Gamma model, the priors for the mean and precision are coupled:

\\[
\lambda_{kd} \sim \operatorname{Gamma}(a_0,b_0),
\\]

\\[
\mu_{kd}\mid\lambda_{kd} \sim
N\left( m_{0d}, \frac{1}{\beta_0\lambda_{kd}} \right).
\\]

Note that, conditional on a large precision, the prior for the component mean becomes concentrated around \(m_{0d}\). 

In the simulation the following priors were chosen somewhat arbitrarily:

\\[
\alpha_0=1, \qquad \beta_0=1, \qquad \boldsymbol m_0=(0,0), \qquad a_0=1, \qquad b_0=1.
\\]

For `VB Lambdas1` \\(\equiv \lambda_{11}\\) we have:

\\[
\mu_{11}=-2, \qquad m_{01}=0, \qquad \lambda_{11} = 1/0.4 = 2.5.
\\]

The combination of a mean far from zero and a large precision is relatively unlikely under the default joint prior. It is therefore likely that the variational posterior *reacts* by shrinking \\(\lambda_{11}\\) downward.

For the chosen parameters, the expected effective size of the first component is
\\( N_1 = 75(0.35) = 26.25 \\), which plays a role in the update equations for the parameters:

\\[
a_{11} = a_0+\frac{N_1}{2},\quad
b_{11} = b_0 + \frac{N_1S_{11}}{2} +
\frac{1}{2}\frac{\beta_0N_1}{\beta_0+N_1} (\bar x_{11}-m_{01})^2.
\\]

Using the values of the data generating process:

\\[
a_{11} = 1 + \frac{26.25}{2} = 14.125,\quad
b_{11} = 1 + \frac{26.25(0.4)}{2} + \frac{1}{2} \frac{26.25}{27.25} (-2)^2 = 8.177.
\\]

The variational posterior mean results then

\\[
E_q[\lambda_{11}] = \frac{a_{11}}{b_{11}} = 1.73,
\\]

which is well below the true value \\(\lambda_{11}=2.5\\).

The approximate \(95\%\) variational credible interval at these expected sufficient statistics is \\([0.947, 2.738]\\) (the quantiles of the Gamma distribution, see below).

The true value turns out to be close to the upper limit of the interval. Due to sampling variation, the posterior for `VB Lambdas1` is often shifted sufficiently downward that the upper credible limit falls below the fixed true value 2.5. This explain (at least partly) the observed undercoverage in `VB Lambdas1`.

In [21]:
round(qgamma(c(0.025, 0.975), shape=14.125, rate=8.177), 3)

[1] 0.947 2.738

#### Modify the prior

The example worked out above shows that, in the simulation of frequentist coverages, it is possible that a prior performs poorly for some parameter, even if the Bayesian procedure is internally coherent. This suggests trying different priors.

The prior \\(\beta_0 = 1\\) may be stronger than we could thought at first. Trying a weaker prior \\(\beta_0 = 0.01\\), the coverage of `VB Lambdas1` improves. As shown below, the frequentist coverage increases to 0.952, larger than 0.721 obtained before with \\(\beta_0 = 1\\).

In [22]:
tmp <- evalq({
tab <- rbind(colMeans(do.call("rbind", lapply(res_list, function(x) x$coverage)), na.rm=TRUE))
tab <- rbind(tab[,startsWith(colnames(tab), "EM")], tab[,startsWith(colnames(tab), "VB")])
colnames(tab) <- names(dgp_vec_pars)
rownames(tab) <- c("EM", "VB")
round(tab, 3)
}, envir = env2)
tmp

,pis1,pis2,mus1,mus2,mus3,mus4,Lambdas1,Lambdas2,Lambdas3,Lambdas4
EM,0.954,0.954,0.928,0.942,0.930,0.942,0.924,0.929,0.936,0.939
VB,0.952,0.952,0.948,0.947,0.946,0.951,0.952,0.957,0.945,0.954


#### Predictive density

The previous discussion suggests that just comparing the confidence intervals as described above, does not not sufficiently reveal the conceptual differences between the classical and Bayesian approaches.

In order to extend the original design of the simulation, we will consider further aspects of the procedures. This subsection focuses on predictive densities.

**EM gives a plug-in predictive density**, i.e. a density where the parameter estimate is directly plugged into the model:

\\[
p_{\mathrm{EM}}(\mathbf{x}_{\mathrm{new}}) = p(\mathbf{x}_{\mathrm{new}}\mid \widehat{\theta}_{\mathrm{EM}}).
\\]

**VB provides an approximate posterior predictive density** that integrates over uncertainty in the parameters:

\\[
p_{\mathrm{VB}}(\mathbf{x}_{\mathrm{new}}) =
\int p(\mathbf{x}_{\mathrm{new}}\mid\theta) q(\theta) \,d\theta.
\\]

In the simulation the log-predictive density (LPD) was evaluated at each iteration as follows:

- A training sample of size \\(N\\) (the same as in the DGP) is generated.
- The model is fitted to the same training sample by means of EM and VB.
- An independent test sample of size \\(N_{\mathrm{test}}\\) (here the same sample size as the data generating process was used) is generated from the same model.</li>
- The LPD is evaluated on the test sample.

    For EM, the log-predictive density is:

    \\[
    \operatorname{LPD}_{\mathrm{EM}} = \frac{1}{N_{\mathrm{test}}} 
    \sum_{i=1}^{N_{\mathrm{test}}} 
    \log p
    \left(
    \mathbf{x}_i^{\mathrm{test}} \mid
    \widehat{\boldsymbol{\theta}}_{\mathrm{EM}}
    \right), \qquad N_{\mathrm{test}}=75.
    \\]

    For VB, the posterior predictive density is approximated using \(B\) parameter draws from the variational posterior:

    \\[
    \overline{\operatorname{LPD}}_{\mathrm{VB}} =
    \frac{1}{N_{\mathrm{test}}}
    \sum_{i=1}^{N_{\mathrm{test}}}\log
    \left[
    \frac{1}{B} \sum_{b=1}^{B} 
    p\left( \mathbf{x}_i^{\mathrm{test}} \mid \boldsymbol{\theta}^{(b)} \right)
    \right],
    \\]
    
    \\[
    \boldsymbol{\theta}^{(b)} \sim q(\boldsymbol{\theta}), \qquad N_{\mathrm{test}}=75, \qquad B=1000.
    \\]
    
    \\(N_{\mathrm{test}}\\) is the number of independent test observations over which the log-predictive scores are averaged. In the current simulation, \\(N_{\mathrm{test}}=75 \\).

    The value \\(B\\) is the number of parameter draws from the fitted variational posterior used to approximate the posterior predictive density for each test observation. In the current implementation, \\( B=1000 \\).

    Thus, each of the 75 test observations is evaluated under 1,000 parameter draws, and the resulting conditional densities are averaged before taking the logarithm.

    EM has no corresponding value of \\(B\\), because its predictive density uses the single plug-in estimate \\(\widehat{\boldsymbol{\theta}}_{\mathrm{EM}}\\), rather than integrating over a distribution of parameter values.

In [23]:
pred1 <- evalq({
#res_list[[1]]$predictive
round(t(rbind(colMeans(do.call("rbind", lapply(res_list, function(x) x$predictive))))), 3)
}, envir = env1)

pred2 <- evalq({
#res_list[[1]]$predictive
round(t(rbind(colMeans(do.call("rbind", lapply(res_list, function(x) x$predictive))))), 3)
}, envir = env2)

matrix(cbind(pred1, pred2), ncol = 2, 
  dimnames=list(rownames(pred1), c("prior_v1", "prior_v2")))

,prior_v1,prior_v2
EM_test_lpd_mean,-3.113,-3.113
VB_test_lpd_mean,-3.108,-3.098
VB_minus_EM_test_lpd_mean,0.006,0.015


In [30]:
pc <- evalq({
do.call("rbind", lapply(res_list, function(x) x$predictive))
}, envir = env2)

#round(t(apply(pc, 2, summary)), 3)
cat(sprintf("Proportion of cases where predictive VB outperforms EM: %.2f%%.", 
  mean(pc[, "VB_minus_EM_test_lpd_mean"] >= 0) * 100))

Proportion of cases where predictive VB outperforms EM: 63.28%.

Note: The log-predictive density (LPD) for the EM does not depend on any prior. The table above just repeats the same value for comparison.

Under the weaker prior `prior_v2` we observe a mean difference in the log-predictive density equal to:

\\[
\overline{\operatorname{LPD}}_{\mathrm{VB}} - 
\operatorname{LPD}_{\mathrm{EM}} = -3.098 - (-3.113) = 0.015.
\\]

Exponentiation recovers the density scale: \\( e^{0.015}\approx 1.015 \\). Thus VB assigns around 1.5% more predictive density to a typical test observation. VB outperforms EM predictively in 63.28% of the test samples.

Higher predictive density means that the test observations are, on average, slightly more compatible with the VB predictive distribution than with the EM plug-in predictive density. This favours the performance of VB, but the effect is small. 

Compared to the default prior `prior_v1`, the weaker prior `prior_v2` also improves the mean VB predictive density : \\(e^{-3.098-(-3.108)} \approx 1.010 \\).

These values are intended for illustration in the context the present simulation. The generalization of these results would require a larger simulation, considering a wide range of generating processes.

#### Width of confidence intervals

The EM and VB intervals for the mixing probabilities have nearly identical mean widths, approximately \\(0.21\\). The VB intervals for the component means are generally somewhat wider than the corresponding EM intervals, although the differences become smaller under the weaker prior. For example, the mean width for the first component mean decreases from 0.601 under the default prior to 0.528 under the weaker prior, compared with 0.483 for EM.

The largest differences are observed in the precision parameters. VB generally produces narrower intervals than EM: for \\(\lambda_{11}\\) (VB7), the mean widths are 1.853 under the default prior and 2.551 under the weaker prior, compared with 3.425 for EM. The increase under the weaker prior reflects the reduction in prior information. We observe therefore that a narrower interval is not necessarily advantageous if adequate coverage is not achieved.

In [7]:
tab1 <- evalq({
  tab <- apply(do.call("rbind", lapply(res_list, function(x) x[["widths"]])), 2, summary)
}, envir = env1)
round(tab1, 3)

tab2<- evalq({
  tab <- apply(do.call("rbind", lapply(res_list, function(x) x[["widths"]])), 2, summary)
}, envir = env2)
round(tab2, 3)

,EM1,EM2,EM3,EM4,EM5,EM6,EM7,EM8,EM9,EM10,VB1,VB2,VB3,VB4,VB5,VB6,VB7,VB8,VB9,VB10
Min.,0.168,0.168,0.236,0.322,0.281,0.248,0.933,0.582,0.452,0.704,0.165,0.165,0.353,0.369,0.373,0.276,0.756,0.523,0.433,0.661
1st Qu.,0.209,0.209,0.422,0.460,0.594,0.396,2.602,1.263,0.960,1.226,0.207,0.207,0.541,0.491,0.635,0.416,1.604,1.086,0.808,1.115
Median,0.214,0.214,0.473,0.497,0.664,0.428,3.171,1.541,1.101,1.410,0.211,0.211,0.591,0.526,0.699,0.447,1.828,1.287,0.909,1.264
Mean,0.213,0.213,0.483,0.503,0.674,0.431,3.425,1.669,1.142,1.462,0.210,0.210,0.601,0.529,0.711,0.449,1.853,1.350,0.931,1.297
3rd Qu.,0.218,0.218,0.533,0.540,0.742,0.462,3.943,1.917,1.286,1.646,0.216,0.216,0.651,0.564,0.775,0.480,2.070,1.550,1.034,1.446
Max.,0.261,0.261,1.131,0.924,1.157,0.690,13.796,19.327,2.842,3.574,0.221,0.221,1.062,0.724,1.208,0.702,3.444,4.432,1.789,2.838


,EM1,EM2,EM3,EM4,EM5,EM6,EM7,EM8,EM9,EM10,VB1,VB2,VB3,VB4,VB5,VB6,VB7,VB8,VB9,VB10
Min.,0.168,0.168,0.236,0.322,0.281,0.248,0.933,0.582,0.452,0.704,0.165,0.165,0.295,0.342,0.359,0.269,0.848,0.539,0.427,0.670
1st Qu.,0.209,0.209,0.422,0.460,0.594,0.396,2.602,1.263,0.960,1.226,0.206,0.206,0.472,0.471,0.629,0.412,2.071,1.132,0.872,1.141
Median,0.214,0.214,0.473,0.497,0.664,0.428,3.171,1.541,1.101,1.410,0.211,0.211,0.520,0.507,0.697,0.444,2.462,1.351,0.996,1.301
Mean,0.213,0.213,0.483,0.503,0.674,0.431,3.425,1.669,1.142,1.462,0.210,0.210,0.528,0.510,0.709,0.446,2.551,1.428,1.027,1.339
3rd Qu.,0.218,0.218,0.533,0.540,0.742,0.462,3.943,1.917,1.286,1.646,0.216,0.216,0.576,0.547,0.775,0.477,2.919,1.645,1.150,1.499
Max.,0.261,0.261,1.131,0.924,1.157,0.690,13.796,19.327,2.842,3.574,0.221,0.221,0.953,0.724,1.220,0.706,6.634,6.572,2.214,3.023


#### Tests on parameters

The VB posterior probabilities allows us to make probabilistic statements about comparisons between parameters. In the simulations, the posterior probability

\\[
\Pr_q(\pi_2 \gt \pi_1\mid\mathbf{x})
\\]

has a median of 0.996, a mean close to 0.969, and a first quartile around 0.975. Thus, in most fitted samples, VB assigns very strong probability to the correct ordering of the mixture weights. Nevertheless, the minimum value of 0.151 shows that on occasional samples, evidence contrary to the right ordering is found.

The probabilities

\\[
\Pr_q(\mu_{21}\gt\mu_{11}\mid\mathbf{x}) 
\qquad\text{and}\qquad
\Pr_q(\mu_{22}\gt\mu_{12}\mid\mathbf{x})
\\]

are 1.000. Since the probabilities are approximated with \(B=5000\) posterior draws and then rounded, this is to be understood as very close to 1 (rather than exactly 1).

The clear separation between the means of the generating components therefore makes these comparisons particularly easy to discern. 

These results provide also evidence that label switching is handled correctly in the implementation.

In [31]:
#round(colMeans(do.call("rbind", lapply(res_list, function(x) x$vb_pars_probs))), 3)
tab1 <- evalq({
  tab <- t(apply(do.call("rbind", lapply(res_list, function(x) x$vb_pars_probs)), 2, summary))
}, envir = env1)
round(tab1, 3)

tab2 <- evalq({
  tab <- t(apply(do.call("rbind", lapply(res_list, function(x) x$vb_pars_probs)), 2, summary))
}, envir = env2)
round(tab2, 3)

,Min.,1st Qu.,Median,Mean,3rd Qu.,Max.
VB_Pr_pi2_gt_pi1,0.151,0.974,0.996,0.968,0.999,1
VB_Pr_mu21_gt_mu11,1.000,1.000,1.000,1.000,1.000,1
VB_Pr_mu22_gt_mu12,1.000,1.000,1.000,1.000,1.000,1


,Min.,1st Qu.,Median,Mean,3rd Qu.,Max.
VB_Pr_pi2_gt_pi1,0.151,0.975,0.996,0.969,0.999,1
VB_Pr_mu21_gt_mu11,1.000,1.000,1.000,1.000,1.000,1
VB_Pr_mu22_gt_mu12,1.000,1.000,1.000,1.000,1.000,1


- The choice of the prior can affect uncertainty quantification even when EM and VI point estimates become similar under weaker priors. 
- Mean-field variational inference may underestimate posterior uncertainty, producing intervals that are too concentrated.

The simulation provides a real experience on the overall idea that priors matters that could be difficult to gain in a real application where the true generating model is unknown and the implementation of the algorithms is not explore explicitly.
Unintentionally, the simulation provided a concrete illustration of the importance of prior specification. This is insight would be difficult to obtain in a real application, where the true data generating process is unknown.

### Save current workspace / Load saved workspace

In order to avoid running the whole simulation to update the results displayed above, the workspace after running the simulations are stored in an .RData file. These workspaces can be loaded to recover the results after running a simulation. Then the cells in the Results section can be executed to update the results for the loaded workspace.

This is also useful when the kernel restarts or crashes; just loading the workspace containing the simulation results (and reloading the required packages) allows us resuming the analysis of results.

In [15]:
#save.image(file = "./sim-ci-prior-1.RData")

#save.image(file = "./sim-ci-prior-2.RData")

In [1]:
#env1 <- new.env(parent = globalenv())
#load(file = "./sim-ci-prior-1.RData", envir = env1)
#length(env1$res_list)

#env2 <- new.env(parent = globalenv())
#load(file = "./sim-ci-prior-2.RData", envir = env2)
#length(env2$res_list)

### Conclusions

- The frequentist (fixed-parameter) coverage of VB intervals can be very sensitive to the prior. Priors should not therefore be chosen arbitrarily. In the absence of prior knowledge, even if the prior values look at first glance uninformative, it may have a relevant effect on the results.

- Comparing the confidence intervals as described above, does not not sufficiently reveal the conceptual differences between the classical and Bayesian approaches. Comparison of predictive densities and ilustration of direct tests on parameters allowed by VB illustrated better the scope of both approaches.

- A positive-definite observed information matrix does not guarantee good finite-sample coverage. The results in this simulation reflect to some extent the limitations of Wald intervals in finite samples and the ability of the Student-\\(t\\) and Gamma variational marginals to represent finite sample distributions.
- Interval width should not be interpreted independently of coverage: A narrower interval is advantageous only when it achieves good coverage.

- The results obtained here cannot be generalized to the overall performance of EM and VB. Rather, they should be interpreted within the limited scope of this simulation exercise, which was intended mainly to test the implementation and experiment with the elements involved in both methods.

**For future**

The simulation presented here restricts to the baseline methods. Bootstrapped confidence intervals could also be considered for EM.